# Test prepare evaluation on Colab to leverage CUDA GPU


## Setup Repo

- The project has previously been imported via github, and the data folder with additional files uploaded manually
- The project is located under `drive/MyDive/project/ms-project`


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd drive/MyDrive/project/ms-project

## Install dependencies

The original repo is using uv, but it is not suited to run on Colab with the Cuda devices so we install dependencies with pip instead


In [ ]:
!pip install torch
!pip install numpy
!pip install pillow
!pip install tqdm
!pip install colpali-engine
!pip install python-dotenv
!pip install pdf2image
!pip install datasets
!pip install mteb
!pip install ir-measures
!pip install einops
!pip install transformers
!pip install ollama
!pip install aiohttp
!pip install psutil
!pip install colab-xterm


## Run Ollama for the generation model

Ollama needs to run for the generation model qwen2.5vl:7b to be used, the following commands need to be entered in the terminal created below.

- `curl https://ollama.ai/install.sh | sh`
- `ollama serve &`
- `ollama pull qwen2.5vl:7b`


In [ ]:
%load_ext colabxterm
%xterm

## Reload imports

If a py file is edited, it won't import the revised version but the cache one.

This script enforces an import reload


In [ ]:
import importlib
import sys


def recursive_reload(package_name):
    """Recursively reload all modules in a given package. Useful for Colab or Jupyter after editing .py files."""
    modules_to_reload = [name for name in sys.modules if name.startswith(package_name)]

    for module_name in sorted(modules_to_reload, key=len, reverse=True):
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded: {module_name}")

## Load the test script


In [ ]:
%cd src

In [ ]:
import json
import os
import sys
from pathlib import Path

from datasets import Dataset, DatasetDict
from dotenv import load_dotenv

# Add src path for imports
src_path = Path.cwd()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

load_dotenv()

from evaluation.evaluator import Evaluator
from pipeline.rags.factory_rag import RAGFactory


async def _run_evaluation(
    rag_configs_file: str,
    evaluation_name: str,
):
    """
    Run evaluations on a list of datasets.

    Args:
    - rag_configs_file (str): The rag configs to load for the evaluation.
    - evaluation_name (str): The name of the evaluation to run.

    """
    # Load RAG configurations
    rag_configs_path = src_path / "configs" / f"{rag_configs_file}.json"
    if not rag_configs_path.exists():
        print(f"RAG configs file not found: {rag_configs_path}")
        return

    with rag_configs_path.open("r") as f:
        rag_configs = json.load(f)

    metrics_path = src_path / "evaluation/results/metrics.json"
    metrics_path.mkdir(parents=True, exist_ok=True)
    metrics = {}
    if metrics_path.exists():
        with metrics_path.open("rb") as f:
            metrics = json.load(f)

    try:
        print(f"Running {evaluation_name} on {rag_configs['name']}...")
        if rag_configs["name"] not in metrics:
            metrics[rag_configs["name"]] = {}
        if evaluation_name not in metrics[rag_configs["name"]]:
            metrics[rag_configs["name"]][evaluation_name] = {}

        metrics[rag_configs["name"]][evaluation_name] = await _evaluate(rag_configs)

        # Save after each dataset computed
        with metrics_path.open("w") as f:
            json.dump(metrics, f)

    except Exception as e:
        print(f"Error running {evaluation_name} on {rag_configs['name']}: {e}")
        return

    print(f"Metrics saved to {metrics_path}")


async def _evaluate(rag_configs: dict):
    """
    Evaluate a pretrained model on a dataset.

    Args:
    - rag_configs (dict): The RAG configurations.

    Returns:
    - pandas.DataFrame: A dataframe containing the evaluation metrics.

    """
    rags_data_dir = os.getenv("RAGS_DATA_DIR")
    if rags_data_dir is None:
        # Fallback to default if environment variable is not set
        rags_data_dir = str(src_path / "data/rags")
        print(f"RAGS_DATA_DIR not set, using default: {rags_data_dir}")

    data_dir = Path(rags_data_dir)

    ds = None
    try:
        dataset_name = rag_configs["configs"]["knowledge_base"]
        dataset_json_path = (
            src_path / f"data/evaluation/datasets/{dataset_name}/dataset.json"
        )
        if not dataset_json_path.exists():
            raise FileNotFoundError(
                f"Dataset JSON file not found: {dataset_json_path}",
            )

        with dataset_json_path.open("r") as f:
            dataset = json.load(f)

        corpus_columns = ["id", "image", "doc-id", "corpus-id"]
        queries_columns = ["query-id", "query", "query-type"]
        qrels_columns = ["corpus-id", "query-id", "answer", "score"]

        # Convert list of lists into column-wise dicts
        corpus_data = {
            col: [row[col] for row in dataset["corpus"]] for col in corpus_columns
        }
        queries_data = {
            col: [row[col] for row in dataset["queries"]] for col in queries_columns
        }
        qrels_data = {
            col: [row[col] for row in dataset["qrels"]] for col in qrels_columns
        }

        ds = DatasetDict(
            {
                "corpus": Dataset.from_dict(corpus_data),
                "queries": Dataset.from_dict(queries_data),
                "qrels": Dataset.from_dict(qrels_data),
            },
        )
    except Exception as e:
        print(f"Error loading custom dataset {dataset_name}: {e}")
        raise ValueError(f"Custom dataset {dataset_name} could not be loaded.")

    # Setup retriever and evaluator
    print("Setting up RAGFactory and Evaluator...")
    evaluation_rag = RAGFactory.create_rag(rag_configs, data_dir)
    evaluator = Evaluator(evaluation_rag)
    print("RAGFactory and Evaluator set up successfully.")

    return await evaluator.evaluate_dataset(
        ds=ds,
        k=100,
        batch_size=10,
        complexity="v1",
    )


async def main(rag_configs: str, evaluation_name: str):
    """
    Main function to evaluate a dataset with the retriever.

    Usage:
    uv run evaluation/evaluate.py --rag-configs "multimodal" --evaluation-name "default"
    """
    print("Evaluation started...")

    await _run_evaluation(rag_configs, evaluation_name)

    print("Evaluation completed.")


## Run


In [ ]:
await main("multimodal_arxiv", "multimodal_arxiv")